In [5]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.preprocessing import StandardScaler
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, roc_auc_score, roc_curve
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import TimeSeriesSplit

In [6]:
#Dynamic search path configuration
import sys
import os

#Adjusting search path based on notebook's depth in directory --> iterates through directories until it finds directory name: US_conflict_forecasting
current_dir = os.getcwd()
while 'US_conflict_forecasting' not in os.path.basename(current_dir):
    current_dir = os.path.dirname(current_dir)

#Adjusting search path based on 'new' current directory
if current_dir not in sys.path:
    sys.path.insert(0, current_dir)

from project_setup import setup_project
project_root_dir = setup_project()

import pandas as pd

### This script will run a Random Forest Regression and an XGBoost model

### Load the data

In [7]:
# Load the data

data = pd.read_csv(os.path.join(project_root_dir, 'Dataset/acs_conflict_combined_cleaned.csv'))

#data = pd.read_csv(project_root_dir, 'acs_conflict_combined_cleaned.csv')

## 1. Random Forest

### Features and labels, cros-validation, split and normalization

In [8]:
# Define features and label

X = data.drop(['event_count', 'ZCTA', 'Month'], axis=1)
Y = data['event_count']

In [10]:
# Time Series Cross Validation, 5 folds.

tscv = TimeSeriesSplit(n_splits=5)

In [11]:
# Split the dataset into training and test sets
# 70/30 split

X_train, X_test, Y_train, Y_test = train_test_split(X, Y, test_size=0.3, random_state=1)

In [12]:
# Normalize the data
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

### Define the regressor and run

In [13]:
rf_regressor = RandomForestRegressor(n_estimators=100, random_state=100)

In [14]:
rf_regressor.fit(X_train, Y_train)

RandomForestRegressor(random_state=100)

### Feature importance

In [ ]:
# Which are the most important features?


feature_importances = rf_regressor.feature_importances_

sorted_indices = np.argsort(feature_importances)[::-1]

# Create the plot
plt.figure(figsize=(10, 8))
plt.title('Feature Importances', fontsize=20)
plt.bar(range(len(feature_importances)), feature_importances[sorted_indices], align='center', alpha=0.5)
plt.xticks(range(len(feature_importances)), feature_names[sorted_indices], rotation=90)
plt.ylabel('Relative Importance', fontsize=15)
plt.xlabel('Feature', fontsize=15)
plt.tight_layout()  # Adjusts the plot to ensure everything fits without overlapping
plt.show()

### Run Grid Search to find the best parameters

In [22]:
# Grid Search for best parameters

# Adjust number of trees in forest, max depth of trees, and minimum samples per leaf

# n_estimators: Number of trees in the forest, to take into account for the prediction
# max_depth: Maximum depth of the tree. From root to leaf, basically controls complexity of the tree and can lead/prevent overfitting. The more splits, the more complex the tree and the more likely to fit noise.
# min_samples_split: Minimum number of samples required to split an internal node


# Prevents overfitting and improves the generalization of the model

param_grid = {
    'n_estimators': [10, 50, 100],
    'max_depth': [None, 10, 20, 30],
    'min_samples_split': [2, 5, 10]
}

In [23]:
cv_rf = GridSearchCV(estimator=rf_regressor, param_grid=param_grid, cv=5)
cv_rf.fit(X_train_scaled, Y_train)

## Run the predictions

In [15]:
Y_pred = rf_regressor.predict(X_test_scaled)

predictions = pd.DataFrame({'Actual': Y_test, 'Predicted': Y_pred})
print(Y_pred)

[0.006203   0.00655349 0.00702031 ... 0.4309429  0.00629923 0.006203  ]


### Show predictions and model evaluations

In [16]:
ZIP_results = pd.merge(predictions, data[['ZCTA', 'Month']], left_index=True, right_index=True)


In [17]:
print(ZIP_results)

       Actual  Predicted     ZCTA  Month
2232      0.0   0.006203  15364.0      1
13760     0.0   0.006553  17702.0      9
559       0.0   0.007020  15061.0      8
7310      0.0   0.006203  16238.0      3
352       0.0   0.006414  15042.0      5
...       ...        ...      ...    ...
13909     0.0   0.006203  17742.0      2
10454     0.0   0.006152  16922.0      3
20010     0.0   0.430943  19104.0      7
20584     0.0   0.006299  19301.0      5
14396     0.0   0.006203  17832.0      9

[6599 rows x 4 columns]


In [18]:
ZIP_results = ZIP_results[ZIP_results['Predicted'] >= 1]

In [19]:
mse = mean_squared_error(Y_test, Y_pred)

print('Mean Squared Error:', mse)

Mean Squared Error: 0.07387649930433778


In [ ]:
# Performance metrics
print("Accuracy:", accuracy_score(Y_test, Y_pred))
# print(classification_report(y_test, y_pred))
from sklearn.metrics import confusion_matrix
#from sklearn.metrics import plot_roc_curve
from sklearn.metrics import roc_curve, auc
# Confusion Matrix
cm = confusion_matrix(Y_test, Y_pred)
print("Confusion Matrix:\n", cm)

In [ ]:
# results_df = pd.DataFrame(columns=['Model', 'RMSLE', 'RMSE', 'MSE', 'MAE'])


## 2. XGboost

### Fit the model

In [ ]:
xgb = XGBClassifier(use_label_encoder=False, eval_metric='logloss', random_state= 1 )
xgb.fit(X_train_scaled, Y_train)

### Run Grid Search to find the best parameters:

In [ ]:
param_grid = {
    'max_depth': [3, 4, 5, 6, 7],
    'min_child_weight': [1, 2, 3, 4],
    'learning_rate': [0.01, 0.05, 0.1, 0.2],
    'n_estimators': [50, 100, 150, 200],
    'subsample': [0.6, 0.7, 0.8, 0.9],
    'colsample_bytree': [0.6, 0.7, 0.8, 0.9]
}

In [ ]:
# Initialize XGB and GridSearch
grid_search = GridSearchCV(xgb, param_grid, scoring='accuracy', cv=5, verbose=1)
grid_search.fit(X_train_scaled, Y_train)

# Print the best parameters
best_params = grid_search.best_params_
print("Best parameters:", best_params)

### Re-run the model with optimal paramter values

In [ ]:
# Re-train the model with the best parameters found above
xgb_optimized = XGBClassifier(**best_params, use_label_encoder=False, eval_metric='logloss', random_state = 1)
xgb_optimized.fit(X_train_scaled, Y_train)

# Make predictions
y_pred_optimized = xgb_optimized.predict(X_test_scaled)

# Accurcary score
accuracy_optimized = accuracy_score(Y_test, y_pred_optimized)
print("Test set accuracy with optimized parameters: {:.2f}%".format(accuracy_optimized * 100))

### Run feature importance analysis

In [ ]:
importances = xgb.feature_importances_
indices = np.argsort(importances)[::-1]
plt.figure()
plt.title('Feature Importances')
plt.bar(range(X_train.shape[1]), importances[indices], color='r', align='center')
plt.xticks(range(X_train.shape[1]), X_train.columns[indices], rotation=90)
plt.xlim([-1, X_train.shape[1]])
plt.show()

### Find best number of features

In [ ]:
# Empty list to store and run different cv scores
cv_scores = []

for i in range(1, len(indices) + 1):
    # Select the top 'i' features
    top_features = [X_train.columns[indices[j]] for j in range(i)]
    
    # Create the training data using the selected features
    X_train_top = X_train_scaled[:, indices[:i]]
    
    # Initialize the model with the best parameters
    model = XGBClassifier(**best_params, use_label_encoder=False, eval_metric='logloss', random_state=42)
    
    # Perform cv at 5 and store the mean score
    scores = cross_val_score(model, X_train_top, Y_train, cv=5, scoring='accuracy')
    cv_scores.append(scores.mean())

# Which number of features gives the highest cv score?
optimal_features = np.argmax(cv_scores) + 1 
optimal_score = cv_scores[optimal_features - 1]

print(f"Optimal number of features: {optimal_features}")
print(f"Highest CV accuracy: {optimal_score:.2f}%")

### Plot

In [ ]:
plt.figure(figsize=(10, 6))
plt.plot(range(1, len(indices) + 1), cv_scores, marker='o')
plt.xlabel('Number of Features')
plt.ylabel('CV Accuracy')
plt.title('CV Performance vs. Number of Features')
plt.show()

### Re-run with best features

In [ ]:


# Select opimal features
top_features_indices = indices[:optimal_features]
X_train_optimal = X_train_scaled[:, top_features_indices]
X_test_optimal = X_test_scaled[:, top_features_indices]

# Train the model using the selected subset of features
model_optimal = XGBClassifier(**best_params, use_label_encoder=False, eval_metric='logloss', random_state=42)
model_optimal.fit(X_train_optimal, y_train)

# Evaluate the model on the test set
y_pred_optimal = model_optimal.predict(X_test_optimal)
accuracy_optimal = accuracy_score(y_test, y_pred_optimal)

print(f"Test set accuracy with optimal number of features ({optimal_features}): {accuracy_optimal:.2f}%")